# 데이터 예시
```
anchor,positive
"TOP클래스 종신보험", "무배당 유니버설 종신보험"
"참좋은운전자보험", "운전자 전용 배상책임보험"
"건강보험1904", "의료비보장 건강플랜"
...

```

# E5 / BGE 모델용 파인튜닝 코드 (with SentenceTransformers)

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import pandas as pd
import os

# ✅ Step 1. 모델 지정
# model_name = "intfloat/multilingual-e5-large"
model_name = "BAAI/bge-m3"

# ✅ Step 2. 학습 데이터 로딩
df = pd.read_csv("train.csv")
train_samples = [
    InputExample(texts=[f"query: {row['anchor']}", f"passage: {row['positive']}"])  # E5 형식
    # InputExample(texts=[f"Represent this sentence for clustering: {row['anchor']}",
    #                     f"Represent this sentence for clustering: {row['positive']}"])  # BGE 형식 (선택)
    for _, row in df.iterrows()
]

# ✅ Step 3. 모델 로딩
model = SentenceTransformer(model_name)

# ✅ Step 4. Dataloader & Loss
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=32)
train_loss = losses.MultipleNegativesRankingLoss(model)

# ✅ Step 5. 파인튜닝 수행
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=100,
    output_path="./fine_tuned_e5_or_bge_model"
)

In [ ]:
model = SentenceTransformer("./fine_tuned_e5_or_bge_model")

# E5 모델이라면 query/passage prefix를 다시 붙이세요
query = "query: 참좋은운전자보험"
candidate = "passage: 운전자 배상책임 전용보험"

embeddings = model.encode([query, candidate])

```
- Negative sample이 있다면 TripletLoss, ContrastiveLoss 사용 가능
- 라벨이 있다면 (1/0)	CosineSimilarityLoss로 fine-tuning 가능
- 파인튜닝 평가용 지표	Recall@K, mAP, NDCG@K, Silhouette 등
HuggingFace에서 학습하려면	Trainer + SiameseDataset 구성도 가능 (필요 시 제공 가능)
```

# SimCSE (Unsupervised) 파인튜닝 코드 예시

In [ ]:
from sentence_transformers import SentenceTransformer, models, losses, InputExample
from torch.utils.data import DataLoader
import pandas as pd

# ✅ Step 1. 상품명 리스트 불러오기
df = pd.read_csv("product_names.csv")  # 단일 컬럼 `product_name`
examples = [InputExample(texts=[row['product_name']]) for _, row in df.iterrows()]

# ✅ Step 2. 모델 로딩 (E5나 BGE도 가능하나 일반 SBERT-style이 적합)
model = SentenceTransformer("nlpai-lab/KURE-v1")

# ✅ Step 3. Dataloader
train_dataloader = DataLoader(examples, shuffle=True, batch_size=64)

# ✅ Step 4. SimCSE Loss (unsupervised)
loss = losses.SelfContrastiveLoss(model)

# ✅ Step 5. 파인튜닝
model.fit(
    train_objectives=[(train_dataloader, loss)],
    epochs=1,
    warmup_steps=100,
    output_path="./simcse_finetuned_product_model"
)


In [ ]:
model = SentenceTransformer("./simcse_finetuned_product_model")
embeddings = model.encode(["TOP클래스종신보험", "TOP클래스 종신보험"])